# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")
print(f"Data Collection: {metadata.dataCollection}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets and their `@id`s, as well as the fields and columns within each record set. We will enumerate all record sets and reference by their `@id`.

In [ ]:
# List all record sets by @id
record_sets = []

for rs in metadata.recordSet:
    print(f"RecordSet @id: {rs['@id']} - name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])
    # Print fields
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    Field @id: {field['@id']} - name: {field.get('name', '')} (dataType: {field.get('dataType', '')})")
    # Print columns
    columns = rs.get('column', [])
    if not isinstance(columns, list):
        columns = [columns]
    print("  Columns:")
    for col in columns:
        print(f"    Column @id: {col['@id']} - name: {col.get('name', '')}")
    print()

# Ensure record_sets contains all @id values
print(f"RecordSet IDs: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. Reference entities only by their `@id`.

In [ ]:
# Assuming at least one record set was listed above
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set @id: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}\n")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
        continue

# Choose one record set for further exploration
if record_sets:
    record_set_id = record_sets[0]  # Use the first record set
    print(f"Preview of DataFrame for record set @id: {record_set_id}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Entities must be referenced by their `@id`.

In [ ]:
# EDA: Filtering, normalizing, and grouping
from pandas.api.types import is_numeric_dtype

df = dataframes[record_set_id]

# Find a numeric field referenced by its @id
numeric_field_id = None

for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a categorical field to group by
    group_field = None
    for col in df.columns:
        if not is_numeric_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found in selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
All visualizations reference columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Distribution of numeric column
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of field (@id): {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field is available, boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"Distribution of {numeric_field_id} by {group_field} (both referenced by @id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id`. We:
- Loaded metadata and reviewed dataset context
- Inspected available record sets, fields, and columns
- Extracted and explored records for a chosen record set (referenced by `@id`), performing basic filtering and normalization
- Visualized numeric and categorical relationships in the dataset

This approach ensures reliable, reproducible analysis leveraging the Croissant schema, and prepares the dataset for deeper modeling and insight generation.